# SMS Spam Inference and Error Analysis

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import PipelineModel

In [2]:
spark = SparkSession.builder.appName('sms-spam-inference').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

model = PipelineModel.load('/content/artifacts/models/sms_spam_pipeline')
clean_df = spark.read.parquet('/content/artifacts/clean_sms.parquet')

print('Rows:', clean_df.count())
clean_df.show(5, truncate=False)

Rows: 5572
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|label|label_text|message                                                                                                                                                    |msg_len_chars|msg_len_words|
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|0    |ham       |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                            |111          |20           |
|0    |ham       |Ok lar... Joking wif u oni...                                                                                                                              |29 

In [4]:
from pyspark.ml.functions import vector_to_array

scored_df = model.transform(clean_df.select('label', 'message'))
scored_df = scored_df.withColumn('prob_array', vector_to_array('probability'))
scored_df = scored_df.withColumn('spam_prob', F.col('prob_array')[1])

scored_df.select('label', 'prediction', 'spam_prob', 'message').show(10, truncate=False)

uncertain_df = scored_df.withColumn('uncertainty', F.abs(F.col('spam_prob') - F.lit(0.5))).orderBy('uncertainty')
uncertain_df.select('label', 'prediction', 'spam_prob', 'message').show(15, truncate=False)

+-----+----------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|spam_prob             |message                                                                                                                                                         |
+-----+----------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0    |0.0       |2.6204343139824005E-9 |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                                 |
|0    |0.0       |5.431016747436956E-9  |Ok lar... Joking wif u oni...                                                                                                                      

In [5]:
from pathlib import Path

false_pos = scored_df.filter((F.col('label') == 0) & (F.col('prediction') == 1)).select('spam_prob', 'message')
false_neg = scored_df.filter((F.col('label') == 1) & (F.col('prediction') == 0)).select('spam_prob', 'message')

print('False positives:', false_pos.count())
print('False negatives:', false_neg.count())

artifact_root = Path('/content/artifacts')
false_pos.toPandas().to_csv(artifact_root / 'false_positives.csv', index=False)
false_neg.toPandas().to_csv(artifact_root / 'false_negatives.csv', index=False)
print('Saved error files to:', artifact_root)

false_pos.orderBy(F.desc('spam_prob')).show(10, truncate=False)
false_neg.orderBy(F.asc('spam_prob')).show(10, truncate=False)

False positives: 9
False negatives: 18
Saved error files to: /content/artifacts
+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|spam_prob         |message                                                                                                                                                                                                                                                                                                                           |
+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------